## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


# 自分用コード
ここから

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [4]:
from pathlib import Path

for filename in ["glossary_sft.jsonl", "glossary_eval.json"]:
    path = Path("/content") / filename
    print(filename, "OK" if path.exists() else "NOT FOUND")

glossary_sft.jsonl OK
glossary_eval.json OK


In [5]:
import json
import random
from pathlib import Path
from datasets import Dataset

# 学習データ116件を読み込む
with open("/content/glossary_sft.jsonl", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

# 再現性のためseedを固定
random.Random(42).shuffle(records)

# 学習用と検証用を分離（約9:1）
n_valid = 12
train_records = records[n_valid:]
valid_records = records[:n_valid]

train_ds = Dataset.from_list(train_records)
valid_ds = Dataset.from_list(valid_records)

print("Total:", len(records))
print("Train:", len(train_ds))
print("Validation:", len(valid_ds))
print("Example:", train_ds[0]["messages"])


Total: 116
Train: 104
Validation: 12
Example: [{'role': 'user', 'content': 'コンテキストウィンドウについて簡単に説明してください。'}, {'role': 'assistant', 'content': 'LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。'}]


In [6]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 3,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 3,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 96,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}


In [7]:

import json
import random
from collections import defaultdict
from pathlib import Path
from datasets import Dataset

# 公式用語集を読み込む
glossary = json.loads(
    Path("/content/glossary.json").read_text(encoding="utf-8")
) if Path("/content/glossary.json").exists() else None

# 学習用JSONLを読み込む
records = [
    json.loads(line)
    for line in Path("/content/glossary_sft.jsonl")
        .read_text(encoding="utf-8").splitlines()
    if line.strip()
]

# 同じ定義を持つQ&Aをグループ化
# 公式スクリプトでは各用語につき2質問・同一回答を生成
groups = defaultdict(list)

for record in records:
    answer = record["messages"][1]["content"]
    groups[answer].append(record)

# 用語単位でシャッフル
group_keys = sorted(groups.keys())
random.Random(42).shuffle(group_keys)

# 58用語のうち6用語をValidationに割り当て
valid_keys = set(group_keys[:6])

train_records = []
valid_records = []

for key in group_keys:
    if key in valid_keys:
        valid_records.extend(groups[key])
    else:
        train_records.extend(groups[key])

train_ds = Dataset.from_list(train_records)
valid_ds = Dataset.from_list(valid_records)

# 重複チェック
train_answers = {
    row["messages"][1]["content"]
    for row in train_records
}
valid_answers = {
    row["messages"][1]["content"]
    for row in valid_records
}

assert train_answers.isdisjoint(valid_answers)
assert len(train_records) + len(valid_records) == 116

print("Total:", len(records))
print("Train:", len(train_ds))
print("Validation:", len(valid_ds))
print("Train terms:", len(train_answers))
print("Validation terms:", len(valid_answers))
print("Data leakage check: PASSED")


Total: 116
Train: 104
Validation: 12
Train terms: 52
Validation terms: 6
Data leakage check: PASSED


In [8]:
print("=== Held-out Validation Terms ===")

for i, row in enumerate(valid_records[::2], 1):
    print(f"\n{i}. {row['messages'][0]['content']}")
    print(f"   正解: {row['messages'][1]['content']}")

=== Held-out Validation Terms ===

1. 「mcpServers設定」とは何ですか？
   正解: 使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。

2. 「外部ツール」とは何ですか？
   正解: Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。

3. 「AIエンジニアリング」とは何ですか？
   正解: 学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。

4. 「コンテキストウィンドウ」とは何ですか？
   正解: LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。

5. 「ハーネス」とは何ですか？
   正解: LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。

6. 「MCPクライアント」とは何ですか？
   正解: MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプ

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 形式はプロンプトで指示している。モデルがこれを守れるかが学習前の確認点
SYSTEM = "あなたは講座 FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。FAQ に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。"

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


Evaluation questions: 15
Categories: {'known': 5, 'paraphrase': 5, 'unknown': 5}


In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["base_model"],
    revision=CONFIG["revision"],
    trust_remote_code=True
)

print("Tokenizer loaded:", tokenizer is not None)

Tokenizer loaded: True


In [17]:
import json
import time
import torch
from pathlib import Path

# 評価用の質問を作成
questions = list(eval_questions)

# 学習から除外した6用語を追加
for row in valid_records[::2]:
    questions.append({
        "category": "held_out",
        "question": row["messages"][0]["content"],
        "term_id": None,
        "expected": row["messages"][1]["content"]
    })

assert len(questions) == 21

# 学習前のモデルで回答を生成

def generate_answer(question, target_model):
    messages = [
        {"role": "user", "content": question}
    ]

    # BatchEncodingではなくTensorを取得する
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt"
    )

    # BatchEncodingの場合はinput_idsを取り出す
    if hasattr(inputs, "input_ids"):
        inputs = inputs["input_ids"]

    inputs = inputs.to(target_model.device)

    with torch.inference_mode():
        output = target_model.generate(
            input_ids=inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][inputs.shape[-1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


results_before = []

start = time.perf_counter()

for i, q in enumerate(questions, 1):
    answer = generate_answer(q["question"], model)

    results_before.append({
        **q,
        "answer": answer
    })

    print(f"[{i}/{len(questions)}] {q['category']}: {answer[:80]}")

elapsed = time.perf_counter() - start

output_path = Path("/content/glossary_before.json")
output_path.write_text(
    json.dumps(results_before, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\nBaseline evaluation completed")
print("Questions:", len(results_before))
print(f"Elapsed: {elapsed:.1f} seconds")
print("Saved:", output_path)


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[1/21] known: **LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術です。

簡単に言うと、**「モデルの学習を細かく制御して、特定の知識や文
[2/21] known: **PEFT** は、**P**rompt **E**valuation **T**est **F**orm **T**est（プロンプト評価テスト形式）の略で
[3/21] known: **ファインチューニング**（Fine Tuning）は、ソフトウェアやシステムにおいて、ユーザーの操作や設定をより自然に反映させ、より快適に使いやすくするため
[4/21] known: **過学習（Overfitting）**とは、機械学習や統計学習において、モデルが学習したデータに**過剰に適合**し、**真のデータに適用できない**状態を指
[5/21] known: **量子化**（Quantization）は、物理学において**「量子力学」の核となる概念**です。

一言で言うと、**「連続的な量ではなく、特定の「量子」の
[6/21] paraphrase: LoRA（Lightning-optimized Residual Attention）の更新と固定（または固定化）された重みの違いは、**「学習の速度と精度の
[7/21] paraphrase: モデルの一部のパラメータだけを学習する方法の総称は、**部分最適化（Partial Optimization）**です。

この手法は、全パラメータをすべて学習
[8/21] paraphrase: 既存の Large Language Models (LLM) を特定の用途に適応させるには、**「知識ベースの追加学習（Fine-tuning）」**と**「
[9/21] paraphrase: 訓練データ（学習データ）と初めて見る質問（テストデータ）で性能が落ちる現象は、**「学習データに埋め込まれた知識」が「新しい問題」に適用される際、意外なケースに
[10/21] paraphrase: モデルの重みを少ないビット数で表現する主な目的は、**「計算コストの削減」**と**「学習の安定性向上」**です。

具体的には以下の 3 つの観点から達成され
[11/21] unknown: 第 14 回講

## Baselineをバックアップする
Colabはセッションが切れるとファイルが消えることがあります。LoRA学習を始める前に、学習前の結果をローカルMacに保存しましょう。

In [18]:
from google.colab import files

files.download("/content/glossary_before.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## セクション4を改変


In [20]:
print("Train:", len(train_ds))
print("Validation:", len(valid_ds))
print("Epochs:", CONFIG["num_train_epochs"])
print("Model:", CONFIG["base_model"])
print("Baseline saved:", Path("/content/glossary_before.json").exists())

Train: 104
Validation: 12
Epochs: 3
Model: Qwen/Qwen3.5-0.8B
Baseline saved: True


In [19]:
import time
from pathlib import Path
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# 事前に作成した用語単位の分割を使用
assert len(train_ds) == 104
assert len(valid_ds) == 12

model.config.use_cache = False

lora = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

USE_AMP = False

sft = SFTConfig(
    output_dir=str(WORK / "ckpt"),
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=CONFIG["lr"],
    lr_scheduler_type="cosine",
    warmup_steps=CONFIG["warmup_steps"],
    logging_steps=10,
    save_steps=10,
    save_total_limit=2,
    fp16=USE_AMP,
    max_length=CONFIG["max_seq_len"],
    packing=False,
    report_to="none",
    seed=CONFIG["seed"]
)

trainer = SFTTrainer(
    model=model,
    args=sft,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    peft_config=lora,
    processing_class=tokenizer
)

trainer.model.print_trainable_parameters()

t0 = time.time()
trainer.train()

print(
    f"学習時間: {time.time()-t0:.0f} 秒 | "
    f"ステップ数: {trainer.state.global_step} | "
    f"エポック: {trainer.state.epoch:.1f}"
)

ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)

if USE_DRIVE:
    import shutil
    shutil.copytree(
        ADAPTER_DIR,
        Path(CONFIG["drive_dir"]) / "adapter",
        dirs_exist_ok=True
    )

tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()


Tokenizing train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

... (2 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,3.245668
20,2.038425
30,1.574261


学習時間: 103 秒 | ステップ数: 39 | エポック: 3.0


In [21]:
from pathlib import Path

print("Train:", len(train_ds))
print("Validation:", len(valid_ds))
print("Epochs:", CONFIG["num_train_epochs"])
print("Model:", CONFIG["base_model"])
print("Baseline saved:", Path("/content/glossary_before.json").exists())

print("Training steps:", trainer.state.global_step)
print("Completed epochs:", trainer.state.epoch)
print("Adapter saved:", (WORK / "adapter").exists())

Train: 104
Validation: 12
Epochs: 3
Model: Qwen/Qwen3.5-0.8B
Baseline saved: True
Training steps: 39
Completed epochs: 3.0
Adapter saved: True


## 学習後のモデルを評価する

In [22]:
from pathlib import Path

print("Train:", len(train_ds))
print("Validation:", len(valid_ds))
print("Epochs:", CONFIG["num_train_epochs"])
print("Training steps:", trainer.state.global_step)
print("Baseline saved:", Path("/content/glossary_before.json").exists())
print("Adapter saved:", (WORK / "adapter").exists())
print("Tuned model:", tuned is not None)

Train: 104
Validation: 12
Epochs: 3
Training steps: 39
Baseline saved: True
Adapter saved: True
Tuned model: True


In [23]:

import json
import time
from pathlib import Path

results_after = []
start = time.perf_counter()

tuned.eval()

for i, q in enumerate(questions, 1):
    answer = generate_answer(q["question"], tuned)

    results_after.append({
        **q,
        "answer": answer
    })

    print(
        f"[{i}/{len(questions)}] "
        f"{q['category']}: {answer[:80]}"
    )

elapsed_after = time.perf_counter() - start

output_path = Path("/content/glossary_after.json")

output_path.write_text(
    json.dumps(
        results_after,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("\nPost-training evaluation completed")
print("Questions:", len(results_after))
print(f"Elapsed: {elapsed_after:.1f} seconds")
print("Saved:", output_path)


[1/21] known: 学習の初期化や追加の学習で、重みを増やしすぎないために使う低精度の重み更新法。学習率を下げると学習がゆっくり進み、メモリも消費も減る。
[2/21] known: モデルの全パラメータを、学習するデータセットの1つのサンプルで割って使う手法。モデルの重みはほとんど変化しないので、学習するデータセットが少ければ学習がしやすく
[3/21] known: 入力データと出力のペアを学習する機械学習の追加ステップ。モデルが特定のタスクや文脈でよく使う表現を学習させる。
[4/21] known: 学習データに過ぎないモデル。訓練データで過学習し、検証データで過学習を察して、テストデータで評価し、テストデータに渡す前の段階でモデルを調整する。
[5/21] known: モデルの重みをより小さなビット数で表し、メモリ使用量や計算量を増やす手法。精度は少し落ちるが、学習するデータや出力の精度はほとんど変わらない。
[6/21] paraphrase: LoRAでは、学習する重みが非常に小さい（例：10^-4）。モデルの重みが大きく、LoRAの重みが小さいため、モデルの重みに影響を与えない。LoRAの重みを固定
[7/21] paraphrase: モデルのすべてのパラメータを学習させることとは別に、特定の部分（例：トークン識別器、重み）だけを学習させること。学習するパラメータの数を減らすことで、学習するデ
[8/21] paraphrase: モデルの出力を特定のタスクで確認し、その結果をモデルに返すこと。返す結果が正しいかどうかを評価し、正しいとモデルを更新させる。Pydantic AIではLLMを
[9/21] paraphrase: モデルは学習した知識やパターンに依存するため、新しい質問では答えを返さずに「何かを返す」という指示に従って、学習していない内容や別の知識に答える。指示文の形式や
[10/21] paraphrase: メモリ使用量や計算量、通信量を削減する。1ビットが1つの0や1を表し、2ビットが2つの状態を表すため、同じ情報を2ビットで表すなら1ビットで表す2倍のデータ量に
[11/21] unknown: LLMのコード生成について簡単に説明する。出力の質を上げるために、モデルをいくつかの役割（例：コーディングエージェント、コ

結果保存

In [24]:
from google.colab import files

files.download("/content/glossary_before.json")
files.download("/content/glossary_after.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ここまで


---

